In [1]:
# # Clone the Parrot repository
# !pip install git+https://github.com/PrithivirajDamodaran/Parrot_Paraphraser.git
# !pip install transformers

In [2]:
from difflib import SequenceMatcher
import torch
import difflib
import pandas
import re
import warnings
from parrot import Parrot
import torch
import warnings
from langdetect import detect
from googletrans import Translator
import time

warnings.filterwarnings("ignore")

AttributeError: module 'httpcore' has no attribute 'SyncHTTPTransport'

1. Read file and split text and sentiment

In [65]:
fold = "1"

texts = []
sentiments = []

with open(f"Datasets-5-fold\Original\splited-{fold}.txt", "r", encoding='utf-8') as file:
    for line in file:
        splited = line.split('')
        texts.append(splited[0])
        sentiments.append(splited[1])



2. Detect language and translate TH -> EN

In [66]:
# # Initialize the Translator 
# translator = Translator()

# translated_texts = []
# for sentence in texts:
#     if detect(sentence.strip()) == "th":
#         translated_sentence = translator.translate(sentence.strip(), src='th', dest='en').text
#         translated_texts.append(translated_sentence)
#         # time.sleep(1)
#     else:
#         translated_texts.append(sentence.strip())


4. Use Parrot-Paraphraser to paraphrase sentences 

In [ ]:
warnings.filterwarnings("ignore")

''' 
uncomment to get reproducable paraphrase generations
def random_state(seed):
  torch.manual_seed(seed)
  if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

random_state(1234)
'''

#Init models (make sure you init ONLY once if you integrate this to your code)
parrot = Parrot(model_tag="prithivida/parrot_paraphraser_on_T5")
translator = Translator()

para_phrases = []
AIgen_texts = []

for i, sentence in enumerate(texts):
    try:    # diversity_ranker="levenshtein"
        para_phrases = parrot.augment(input_phrase=sentence, use_gpu=True, diversity_ranker="levenshtein", do_diverse=True ) #diversity_ranker="levenshtein",do_diverse=True, max_return_phrases = 2, max_length=2000 

        if para_phrases is None:
            AIgen_texts.append(texts[i])
        else: 
            AIgen_texts.append(str(para_phrases[0][0]))      

        print(f"{i+1}/{len(texts)}")
        print("Org_phrase:   ", texts[i])
        print("Aug_phrase:   ", AIgen_texts[i])

    except Exception as e:
        print(f"Error occurred during augmentation: {e}")


In [ ]:
with open(f"Parrot-Paraphraser/5-fold/parrot_generate_{fold}.txt", "w", encoding='utf-8') as file:    
    for text in AIgen_texts:
        file.write(f"{text}\n")

Duplicate check

In [ ]:
filtered_PP_text = []
filtered_PP_sentiments = []

duplicate_count = 0

for i, text in enumerate(AIgen_texts):
    if text not in texts:
        filtered_PP_text.append(text)
        filtered_PP_sentiments.append(sentiments[i])
    else:
        duplicate_count += 1  # Count duplicates

with open(f"Parrot-Paraphraser/5-fold/parrot-sentiment_{fold}.txt", 'w', encoding='utf-8') as file:
    for text, sentiment in zip(filtered_PP_text, filtered_PP_sentiments):
        file.write(f"{text}{sentiment}")

with open(f"Parrot-Paraphraser/5-fold/original-sentiment_{fold}.txt", 'w', encoding='utf-8') as file:
    for text, sentiment in zip(texts, sentiments):
        file.write(f"{text}{sentiment}")


Concat files

In [72]:
file1 = open(f"Parrot-Paraphraser/5-fold/parrot-sentiment_{fold}.txt", 'r', encoding='utf-8')
file2 = open(f"Parrot-Paraphraser/5-fold/original-sentiment_{fold}.txt", 'r', encoding='utf-8')

content1 = file1.read()
content2 = file2.read()

file1.close()
file2.close()

final = open(f"Parrot-Paraphraser/5-fold/original-parrot-train_{fold}.txt", 'w', encoding='utf-8')
final.write(content1 + content2)
final.close()

In [73]:
with open(f"Datasets-5-fold\Original\splited-{fold}.txt", 'r', encoding='utf-8') as fp:
    og = len(fp.readlines())
    print('Total lines:', og)
    
with open(f"Parrot-Paraphraser/5-fold/original-parrot-train_{fold}.txt", 'r', encoding='utf-8') as fp:
    x = len(fp.readlines())
    print('Total lines:', x)

Total lines: 161
Total lines: 220


In [74]:
PP = len(filtered_PP_text)
print(f"Train{fold}")
print("Duplicate count:", duplicate_count)
print("Available data from PP: ", PP)
print("Original data: ", og)
print("Total amount (OG+PP): ", x)

Train1
Duplicate count: 102
Available data from PP:  59
Original data:  161
Total amount (OG+PP):  220


------------------------------------------------------------------------------------------------------------------------------------------------------------

# Changing Score

In [ ]:
original_translated_texts = []
original_texts = []
original_sentiments = []
parrot_texts = []
augment_percents = []
average_change = 0

with open("Datasets-5-fold/Original/splited-1.txt", "r", encoding='utf-8') as file:
    for line in file:
        splited = line.split('')  # Ensure this is the correct delimiter
        original_texts.append(splited[0])
        original_sentiments.append(splited[1])

with open("Parrot-Paraphraser/5-fold/splitedData-pp/pp1/parrot_generate_1.txt", "r", encoding='utf-8') as file:
    for line in file:
        parrot_texts.append(line.strip())

if len(original_translated_texts) != len(parrot_texts):
    print("Warning: Original and Parrot text lengths do not match!")

    for i in range(len(original_texts)):
        original = re.sub('\s+', ' ', original_texts[i].strip())
        parrot = re.sub('\s+', ' ', parrot_texts[i].strip())

        augment_percent = round(1 - SequenceMatcher(a=original, b=parrot).ratio(), 2) * 100
        augment_percents.append(augment_percent)

        output = (f"{i+1}\n"
                f"Original: {original}\n"
                f"Parrot Augmented: {parrot}\n"
                f"Percent of change: {augment_percent:.2f}%\n"
                f"{'-' * 100}\n")
        print(output)

    if augment_percents:
        average_change = round(sum(augment_percents) / len(augment_percents), 2)
        print(f"Average Percent of Change: {average_change}%")

In [ ]:
file1 = open('Parrot-Paraphraser/setting3/original-sentiment_setting3.txt', 'r', encoding='utf-8')
file2 = open('Parrot-Paraphraser/setting3/parrot-sentiment_setting3.txt', 'r', encoding='utf-8')
file1 = file1.readlines()
file2 = file2.readlines()
print(f"Original Length: {len(file1)} Parrot Length: {len(file2)}")

Original Length: 806 Parrot Length: 306


# # other

In [ ]:
file1 = []
file1_texts = []
file1_sentiments = []

file2 = []
file2_texts = []
file2_sentiments = []

with open("Parrot-Paraphraser/setting3/original-sentiment_setting3.txt", "r", encoding='utf-8') as file:
    for line in file:
        file1.append(line)
        splited = line.split('')
        file1_texts.append(splited[0])

with open("Parrot-Paraphraser/setting3/final_parrot_setting3.txt", "r", encoding='utf-8') as file:
    for line in file:
        file2.append(line)
        splited = line.split('')
        file2_texts.append(splited[0])